In [181]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,id,sequence,target_profile
0,RNA_00000,CUAUUUCGGUGCUGAUAUUAAUCCGGAGCGAAACAGCAAUUUUCGC...,0.5234 0.5162 0.7053 0.7230 0.4523 0.5194 0.42...
1,RNA_00001,UGGUGCUAUAUCCGUCACGGGAGAUUUGAUCUUAUUCGUAGCCUAU...,0.5016 0.4668 0.4962 0.2912 0.4395 0.4839 0.35...
2,RNA_00002,AUAGAUGCUGUCCCUAUCGUGCAACGCGUAAACUGACAUAUCACCC...,0.8149 0.5758 0.6977 0.7914 0.6522 0.3694 0.60...
3,RNA_00003,UGUUUUGCGAGGCCUGUACUCUUUAAGUUAGUCAAUUAUGGUAGGG...,0.6819 0.6750 0.6510 0.5464 0.5839 0.5235 0.40...
4,RNA_00004,GUCGAUAUCCGUUGUCGUUGAUCGAGGGGCAACAUGCCAAACUUGC...,0.4282 0.3525 0.3847 0.3119 0.3514 0.4480 0.38...


In [182]:
len(df_train.target_profile[0].split())

243

In [183]:
len(df_train.sequence[0])

243

In [184]:
df_train.sequence.str.len().max()

np.int64(256)

In [185]:
lengths = df_test.sequence.str.len()

In [189]:
import torch

num_epochs = 30
batch_size = 32
learning_rate = 0.0005

num_epochs

30

In [190]:
char_dict = {'A': 1, 'U': 2, 'G': 3, 'C': 4}
max_len = 256

def enc_pad(seq):
    seq = list(seq)
    seq = [char_dict[el] for el in seq]
    pad_len = max_len - len(seq)
    seq = seq + [0] * pad_len
    seq = [str(val) for val in seq]
    return ' '.join(seq)

len(enc_pad(df_train.sequence[0]).split())

256

In [191]:
def pad(labels):
    labels = np.array(labels.split()).astype("float")
    labels = [float(label) for label in labels]
    pad_len = max_len - len(labels)
    labels = list(labels) + [0] * pad_len
    labels = [str(label) for label in labels]
    return ' '.join(labels)

len(pad(df_train.target_profile[0]).split())

256

In [192]:
df_train["enc_seq"] = df_train.sequence.apply(enc_pad)
df_test["enc_seq"] = df_test.sequence.apply(enc_pad)

df_train.target_profile = df_train.target_profile.apply(pad)

df_train.head()

,id,sequence,target_profile,enc_seq
0,RNA_00000,CUAUUUCGGUGCUGAUAUUAAUCCGGAGCGAAACAGCAAUUUUCGC...,0.5234 0.5162 0.7053 0.723 0.4523 0.5194 0.424...,4 2 1 2 2 2 4 3 3 2 3 4 2 3 1 2 1 2 2 1 1 2 4 ...
1,RNA_00001,UGGUGCUAUAUCCGUCACGGGAGAUUUGAUCUUAUUCGUAGCCUAU...,0.5016 0.4668 0.4962 0.2912 0.4395 0.4839 0.35...,2 3 3 2 3 4 2 1 2 1 2 4 4 3 2 4 1 4 3 3 3 1 3 ...
2,RNA_00002,AUAGAUGCUGUCCCUAUCGUGCAACGCGUAAACUGACAUAUCACCC...,0.8149 0.5758 0.6977 0.7914 0.6522 0.3694 0.60...,1 2 1 3 1 2 3 4 2 3 2 4 4 4 2 1 2 4 3 2 3 4 1 ...
3,RNA_00003,UGUUUUGCGAGGCCUGUACUCUUUAAGUUAGUCAAUUAUGGUAGGG...,0.6819 0.675 0.651 0.5464 0.5839 0.5235 0.4056...,2 3 2 2 2 2 3 4 3 1 3 3 4 4 2 3 2 1 4 2 4 2 2 ...
4,RNA_00004,GUCGAUAUCCGUUGUCGUUGAUCGAGGGGCAACAUGCCAAACUUGC...,0.4282 0.3525 0.3847 0.3119 0.3514 0.448 0.386...,3 2 4 3 1 2 1 2 4 4 3 2 2 3 2 4 3 2 2 3 1 2 4 ...


In [193]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, df):
        self.df = df
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        seq = self.df.enc_seq.iloc[idx].split()
        labels = self.df.target_profile.iloc[idx].split()
        seq = [int(val) for val in seq]
        labels = [float(label) for label in labels]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(labels, dtype=torch.float32)

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = MyDataset(df_2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

train_dataset[0][0][0]

tensor(2)

In [194]:
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(5, 16)
        self.lstm = nn.LSTM(input_size=16, hidden_size=128, batch_first=True, bidirectional=True)
        self.fc = nn.LazyLinear(1)
    def forward(self, X):
        x = self.embedding(X.long())
        out, _ = self.lstm(x)
        out = self.fc(out).squeeze(-1)
        return out

model = LSTM().to("cuda")
criterion = nn.MSELoss()
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

LSTM(
  (embedding): Embedding(5, 16)
  (lstm): LSTM(16, 128, batch_first=True, bidirectional=True)
  (fc): LazyLinear(in_features=0, out_features=1, bias=True)
)

In [195]:
def train(epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to("cuda"), labels.to("cuda")
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2)
        optim.step()
        total_loss += loss.item()
    if epoch%5 == 0:
        print("Epoch ", epoch)
        print("Training Loss ", total_loss/len(train_loader))

def test(epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to("cuda"), labels.to("cuda")
            loss = criterion(model(inputs), labels)
            total_loss += loss.item()
    if epoch%5 == 0:
         print("Validation Loss ", total_loss/len(test_loader))

In [196]:
for i in range(1, num_epochs+1):
    train(i)
    test(i)

Epoch  5
Training Loss  0.004638155775144696
Validation Loss  0.004409103523357771
Epoch  10
Training Loss  0.003664087787270546
Validation Loss  0.0035087299256701954
Epoch  15
Training Loss  0.003294139515608549
Validation Loss  0.0031608936333213933
Epoch  20
Training Loss  0.0030070354789495466
Validation Loss  0.0029140027399989776
Epoch  25
Training Loss  0.002842619836330414
Validation Loss  0.0027587321601458825
Epoch  30
Training Loss  0.002775158142670989
Validation Loss  0.002705465754843317


In [197]:
class FinalDataset(Dataset):
    def __init__(self, df):
        self.df = df
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        seq = self.df.enc_seq.iloc[idx].split()
        seq = [int(val) for val in seq]
        return torch.tensor(seq, dtype=torch.float32)

final_dataset = FinalDataset(df_test)
final_loader = DataLoader(final_dataset, batch_size=batch_size, shuffle=False)

final_dataset[0][0]

tensor(3.)

In [198]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to("cuda")
        outputs = model(inputs).cpu().numpy()
        preds.extend(outputs)

preds = np.array(preds)
preds.shape

(1000, 256)

In [199]:
def get_string(preds):
    preds = [round(pred, 4) for pred in preds]
    preds = [str(pred) for pred in preds]
    return ' '.join(preds)

get_string(preds[0])

'0.2063 0.3306 0.3191 0.4358 0.466 0.4663 0.4781 0.4672 0.4391 0.5498 0.5855 0.5482 0.5223 0.5161 0.5356 0.456 0.463 0.3747 0.3393 0.3229 0.4257 0.5186 0.6084 0.713 0.6698 0.7335 0.6271 0.5452 0.4322 0.372 0.2541 0.1249 -0.0078 -0.1005 0.1638 0.2458 0.162 0.22 0.423 0.5095 0.5258 0.4605 0.5464 0.4708 0.5521 0.5625 0.5469 0.5465 0.551 0.5578 0.4759 0.4767 0.3848 0.3665 0.309 0.3295 0.3619 0.28 0.4282 0.3345 0.4651 0.5619 0.5605 0.4326 0.4563 0.2871 0.2535 0.368 0.4016 0.4234 0.5531 0.6627 0.5059 0.4299 0.4803 0.4727 0.4538 0.5372 0.6448 0.6368 0.6422 0.6667 0.6339 0.5342 0.5552 0.5241 0.5466 0.4545 0.4318 0.538 0.4575 0.3305 0.2754 0.4634 0.4532 0.4479 0.452 0.5893 0.4678 0.4837 0.4717 0.5629 0.5738 0.5298 0.5081 0.6305 0.6233 0.5289 0.521 0.5154 0.429 0.3066 0.417 0.4417 0.4248 0.4294 0.5526 0.4485 0.4563 0.3706 0.4352 0.2959 0.3457 0.4121 0.4685 0.4178 0.5848 0.6152 0.3824 0.3601 0.4078 0.3747 0.4237 0.4392 0.453 0.3705 0.3795 0.3932 0.3967 0.4009 0.4829 0.4008 0.4658 0.5584 0.6022 0.

In [200]:
final_preds = []

for i in range(len(preds)):
    length = lengths[i]
    real_pred = preds[i][:length]
    final_preds.append(get_string(real_pred))

final_preds[0]

'0.2063 0.3306 0.3191 0.4358 0.466 0.4663 0.4781 0.4672 0.4391 0.5498 0.5855 0.5482 0.5223 0.5161 0.5356 0.456 0.463 0.3747 0.3393 0.3229 0.4257 0.5186 0.6084 0.713 0.6698 0.7335 0.6271 0.5452 0.4322 0.372 0.2541 0.1249 -0.0078 -0.1005 0.1638 0.2458 0.162 0.22 0.423 0.5095 0.5258 0.4605 0.5464 0.4708 0.5521 0.5625 0.5469 0.5465 0.551 0.5578 0.4759 0.4767 0.3848 0.3665 0.309 0.3295 0.3619 0.28 0.4282 0.3345 0.4651 0.5619 0.5605 0.4326 0.4563 0.2871 0.2535 0.368 0.4016 0.4234 0.5531 0.6627 0.5059 0.4299 0.4803 0.4727 0.4538 0.5372 0.6448 0.6368 0.6422 0.6667 0.6339 0.5342 0.5552 0.5241 0.5466 0.4545 0.4318 0.538 0.4575 0.3305 0.2754 0.4634 0.4532 0.4479 0.452 0.5893 0.4678 0.4837 0.4717 0.5629 0.5738 0.5298 0.5081 0.6305 0.6233 0.5289 0.521 0.5154 0.429 0.3066 0.417 0.4417 0.4248 0.4294 0.5526 0.4485 0.4563 0.3706 0.4352 0.2959 0.3457 0.4121 0.4685 0.4178 0.5848 0.6152 0.3824 0.3601 0.4078 0.3747 0.4237 0.4392 0.453 0.3705 0.3795 0.3932 0.3967 0.4009 0.4829 0.4008 0.4658 0.5584 0.6022 0.

In [201]:
df_preds = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.id,
    "answer": final_preds
})

df_preds

,subtaskID,datapointID,answer
0,1,RNA_00000,0.2063 0.3306 0.3191 0.4358 0.466 0.4663 0.478...
1,1,RNA_00001,0.1708 0.3228 0.2633 0.393 0.4546 0.5579 0.573...
2,1,RNA_00002,0.5891 0.6356 0.5255 0.428 0.4508 0.4505 0.346...
3,1,RNA_00003,0.5744 0.545 0.5675 0.5005 0.4771 0.5298 0.452...
4,1,RNA_00004,0.4664 0.498 0.4094 0.5009 0.5396 0.5412 0.536...
...,...,...,...
995,1,RNA_00995,0.6264 0.6374 0.5763 0.5281 0.4468 0.4824 0.39...
996,1,RNA_00996,0.2106 0.2855 0.3339 0.4367 0.4507 0.5513 0.54...
997,1,RNA_00997,0.5473 0.4089 0.4281 0.4621 0.5406 0.6414 0.72...
998,1,RNA_00998,0.7267 0.7742 0.7343 0.662 0.5767 0.4555 0.464...


In [202]:
df_preds.to_csv("submission.csv", index=False)